# Fine-Tuning eines Sprachmodells
## Textklassifikation mit DistilBERT

In diesem Notebook lernst du, wie man ein vortrainiertes Sprachmodell mit **eigenen Daten** auf eine neue Aufgabe anpasst.

Das nennt sich **Fine-Tuning** – das Modell kennt bereits Sprache, wir bringen ihm jetzt eine spezifische Aufgabe bei.

### Unser Ziel
Wir trainieren ein Modell, das **Produktbewertungen** automatisch als *positiv* oder *negativ* klassifiziert.

---
### Was passiert beim Fine-Tuning?

```
Vortrainiertes Modell          Fine-Tuning               Spezialisiertes Modell
(kennt Sprachstruktur)  →  (lernt unsere Aufgabe)  →  (klassifiziert Texte)
        🧠                        📚                          🎯
```

> **Analogie:** Das Modell ist wie ein Abiturient, der Deutsch gut kann. Wir schulen ihn jetzt speziell darin, Kundenrezensionen zu bewerten – er muss nicht mehr von vorne anfangen.

## Schritt 1: Bibliotheken installieren

Wir brauchen drei Pakete:
- **transformers**: HuggingFace-Bibliothek für vortrainierte Modelle
- **datasets**: Verwaltung von Trainingsdaten
- **scikit-learn**: Auswertung der Ergebnisse

In [ ]:
# Installation (nur einmal nötig)
# Das ~ bedeutet: ungefähr diese Version oder neuer
%pip install transformers datasets scikit-learn --quiet

## Schritt 2: Importe & Gerät festlegen

Wir stellen sicher, dass das Modell auf der **CPU** läuft – das ist langsamer als eine GPU, funktioniert aber auf jedem Computer.

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Gerät bestimmen
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verwendetes Gerät: {device}")
print(f"PyTorch Version: {torch.__version__}")

## Schritt 3: Trainingsdaten erstellen

Wir verwenden kleine, selbst erstellte Beispieldaten – das reicht für einen Demo-Durchlauf.

**Labels:**
- `0` = negativ
- `1` = positiv

> 💡 **Aufgabe:** Kannst du unten eigene Beispiele ergänzen? Achte auf eine ausgewogene Anzahl (gleich viele positive und negative Beispiele).

In [ ]:
# Trainingsdaten – Produktbewertungen
train_texts = [
    # Positiv (Label 1)
    "Das Produkt ist wirklich toll, ich bin sehr zufrieden.",
    "Schnelle Lieferung, alles bestens, gerne wieder!",
    "Absolute Empfehlung, top Qualität für den Preis.",
    "Funktioniert einwandfrei, bin begeistert.",
    "Super Artikel, entspricht genau der Beschreibung.",
    "Sehr gute Verarbeitung, macht einen hochwertigen Eindruck.",
    "Bin vollkommen zufrieden, das war ein guter Kauf.",
    "Hervorragende Qualität, schneller Versand, top!",
    # Negativ (Label 0)
    "Leider sehr enttäuschend, Qualität ist mangelhaft.",
    "Das Produkt hält nicht, was es verspricht. Finger weg!",
    "Lieferung hat ewig gedauert und das Produkt war beschädigt.",
    "Komplette Fehlinvestition, nach einer Woche kaputt.",
    "Sehr schlechte Verarbeitung, für diesen Preis eine Frechheit.",
    "Nie wieder! Der Kundenservice ist auch eine Katastrophe.",
    "Entspricht überhaupt nicht der Beschreibung. Enttäuschend.",
    "Qualität weit unter Erwartung, absolut nicht empfehlenswert.",
]

train_labels = [1, 1, 1, 1, 1, 1, 1, 1,   # positiv
                0, 0, 0, 0, 0, 0, 0, 0]    # negativ

# Testdaten – neue, unbekannte Sätze
test_texts = [
    "Wirklich ein tolles Produkt, kann ich empfehlen.",
    "Völlig unbrauchbar, das Geld ist rausgeschmissen.",
    "Gute Qualität, bin zufrieden.",
    "Hat nach drei Tagen aufgehört zu funktionieren.",
]
test_labels = [1, 0, 1, 0]

print(f"Trainingssätze: {len(train_texts)}")
print(f"Testsätze: {len(test_texts)}")
print(f"Beispiel (positiv): {train_texts[0]}")
print(f"Beispiel (negativ): {train_texts[8]}")

## Schritt 4: Tokenizer laden

Sprachmodelle verstehen keinen Text direkt – sie brauchen **Zahlen**.
Der **Tokenizer** wandelt Text in Token-IDs um.

```
"Das ist gut"  →  [101, 1996, 2003, 2204, 102]
     Text               Token-IDs (Zahlen)
```

Wir verwenden **`distilbert-base-multilingual-cased`** – ein mehrsprachiges, kompaktes Modell, das auch Deutsch versteht.

In [ ]:
MODEL_NAME = "distilbert-base-multilingual-cased"

print(f"Lade Tokenizer: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer geladen!")

# Beispiel: Was macht der Tokenizer?
beispiel = "Das Produkt ist super!"
tokens = tokenizer(beispiel)
print(f"\nBeispiel-Text: '{beispiel}'")
print(f"Token-IDs:     {tokens['input_ids']}")
print(f"Decoded zurück: {tokenizer.decode(tokens['input_ids'])}")

## Schritt 5: Daten tokenisieren und als Dataset verpacken

Wir wandeln alle Texte in Token-IDs um und packen alles in ein HuggingFace-`Dataset`-Objekt.

**`padding=True`** → alle Sequenzen werden auf gleiche Länge gebracht (kürzere werden mit Nullen aufgefüllt)  
**`truncation=True`** → sehr lange Texte werden gekürzt  
**`max_length=64`** → kurze Grenze für schnelleres Training auf CPU

In [ ]:
def tokenize(texts, labels):
    """Tokenisiert eine Liste von Texten und gibt ein Dataset zurück."""
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"   # gibt PyTorch-Tensoren zurück
    )
    # Dataset-Objekt erstellen
    dataset = Dataset.from_dict({
        "input_ids":      encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels":         labels
    })
    dataset.set_format("torch")
    return dataset

train_dataset = tokenize(train_texts, train_labels)
test_dataset  = tokenize(test_texts,  test_labels)

print(f"Trainings-Dataset: {train_dataset}")
print(f"\nErster Eintrag (gekürzt):")
print(f"  input_ids shape: {train_dataset[0]['input_ids'].shape}")
print(f"  label:           {train_dataset[0]['labels']}")

## Schritt 6: Modell laden

Wir laden **DistilBERT** mit einem Klassifikationskopf für **2 Klassen** (positiv / negativ).

> Das Modell wird beim ersten Aufruf heruntergeladen (~250 MB). Danach wird es lokal gecacht.

In [ ]:
print(f"Lade Modell: {MODEL_NAME} ...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,          # 2 Klassen: negativ / positiv
    id2label={0: "negativ", 1: "positiv"},
    label2id={"negativ": 0, "positiv": 1}
)

# Parameteranzahl anzeigen
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModell geladen!")
print(f"Gesamte Parameter:    {total_params:,}")
print(f"Trainierbare Param.:  {trainable_params:,}")

## Schritt 7: Training konfigurieren

Wichtige Hyperparameter:

| Parameter | Bedeutung |
|---|---|
| `num_train_epochs` | Wie oft wird der gesamte Datensatz durchlaufen? |
| `learning_rate` | Wie groß sind die Anpassungsschritte? |
| `per_device_train_batch_size` | Wie viele Beispiele werden gleichzeitig verarbeitet? |

> ⚠️ Auf CPU ist Training langsam. Mit diesen Einstellungen dauert es ca. **1–3 Minuten**.

In [ ]:
# Metrik-Funktion für die Auswertung während des Trainings
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

training_args = TrainingArguments(
    output_dir="./ergebnisse",          # Speicherort für Checkpoints
    num_train_epochs=5,                 # 5 Durchläufe über die Daten
    per_device_train_batch_size=4,      # 4 Beispiele pro Schritt (CPU-freundlich)
    per_device_eval_batch_size=4,
    learning_rate=2e-5,                 # Kleiner Lernschritt (Standard für BERT)
    weight_decay=0.01,                  # Regularisierung gegen Overfitting
    eval_strategy="epoch",              # Nach jeder Epoche auswerten
    save_strategy="no",                 # Kein Speichern (spart Zeit)
    logging_steps=10,
    use_cpu=True,                       # Explizit CPU erzwingen
    report_to="none",                   # Kein Logging an externe Dienste
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Training ist konfiguriert und bereit!")

## Schritt 8: Training starten

Jetzt beginnt das eigentliche Fine-Tuning.
Beobachte, wie der **Loss** (Fehler) mit jeder Epoche sinkt – das bedeutet, das Modell lernt!

```
Epoche 1: Loss hoch   → Modell macht noch viele Fehler
Epoche 3: Loss mittel → Modell lernt die Muster
Epoche 5: Loss niedrig → Modell hat die Aufgabe verstanden
```

In [ ]:
print("Starte Training...")
print("(Auf CPU dauert das ca. 1–3 Minuten)\n")

train_result = trainer.train()

print(f"\nTraining abgeschlossen!")
print(f"Trainingszeit: {train_result.metrics['train_runtime']:.1f} Sekunden")
print(f"Finaler Loss:  {train_result.metrics['train_loss']:.4f}")

## Schritt 9: Auswertung

Wir prüfen, wie gut das Modell auf den **Testdaten** abschneidet – also auf Sätzen, die es beim Training *nicht gesehen* hat.

In [ ]:
# Vorhersagen auf den Testdaten
predictions_output = trainer.predict(test_dataset)
predicted_labels = np.argmax(predictions_output.predictions, axis=-1)

label_names = ["negativ", "positiv"]

print("=" * 50)
print("AUSWERTUNG AUF TESTDATEN")
print("=" * 50)
print()
print(classification_report(
    test_labels,
    predicted_labels,
    target_names=label_names
))

# Einzelergebnisse anzeigen
print("Einzelvorhersagen:")
print("-" * 50)
for text, true, pred in zip(test_texts, test_labels, predicted_labels):
    status = "✅" if true == pred else "❌"
    print(f"{status} Erwartet: {label_names[true]:<10}  "
          f"Vorhergesagt: {label_names[pred]:<10}")
    print(f"   Text: {text[:60]}..." if len(text) > 60 else f"   Text: {text}")
    print()

## Schritt 10: Eigene Texte klassifizieren

Jetzt kannst du eigene Texte eingeben und schauen, wie das Modell sie bewertet!

In [ ]:
def klassifiziere(text):
    """Klassifiziert einen einzelnen Text als positiv oder negativ."""
    # Tokenisieren
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=64
    )
    # Vorhersage
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = torch.softmax(outputs.logits, dim=-1)[0]
    predicted_class = torch.argmax(probs).item()
    label_names = ["negativ", "positiv"]
    
    print(f"Text:       '{text}'")
    print(f"Ergebnis:   {label_names[predicted_class].upper()}")
    print(f"Sicherheit: negativ={probs[0]:.1%}  positiv={probs[1]:.1%}")
    print()

# -----------------------------------------------
# Hier eigene Texte ausprobieren!
# -----------------------------------------------
klassifiziere("Das war das beste Produkt, das ich je gekauft habe!")
klassifiziere("Absolut nutzlos, eine totale Zeitverschwendung.")
klassifiziere("Naja, geht so. Nicht besonders gut, aber auch nicht schlecht.")  # Grenzfall!

## Reflexion & Aufgaben

---

### 📝 Aufgabe 1 – Daten erweitern
Ergänze in **Schritt 3** mindestens 5 eigene Beispiele (positiv und negativ). Führe das Training erneut durch. Verbessert sich die Genauigkeit?

### 📝 Aufgabe 2 – Hyperparameter
Ändere in **Schritt 7** die Anzahl der Epochen (`num_train_epochs`) auf 2 bzw. 10. Was beobachtest du beim Loss? Was ist **Overfitting**?

### 📝 Aufgabe 3 – Grenzfälle
Teste in **Schritt 10** Sätze, die nicht eindeutig positiv oder negativ sind (z.B. ironische Aussagen). Wie verhält sich das Modell?

### 📝 Aufgabe 4 – Übertragung
Fine-Tuning wird in der Praxis für viele Aufgaben genutzt: Spam-Erkennung, Medizinische Texte, Kundensupport-Kategorisierung. Nenne **zwei weitere Anwendungsfälle** aus deinem Alltag.

---

### 💡 Zusammenfassung

| Schritt | Was passiert? |
|---|---|
| Tokenisierung | Text → Zahlen |
| Modell laden | Vorwissen aus Millionen Texten |
| Fine-Tuning | Anpassung an unsere spezifische Aufgabe |
| Evaluation | Überprüfung auf neuen Daten |
| Inferenz | Einsatz auf beliebigen neuen Texten |